In [13]:
import os
from dotenv import load_dotenv, find_dotenv
from pymongo import MongoClient
import pandas as pd
from pymongo.server_api import ServerApi


# Load environment variables
load_dotenv(find_dotenv())

# Connect to MongoDB
mongo_uri = os.environ['MONGO_URI']
client = MongoClient(mongo_uri, server_api=ServerApi('1'))

# Select the database and collection
cm_env = os.environ.get('CM_ENV', 'dev')
if cm_env == 'prod':
    collection = client["campomaq"]["cm_ventas_prods"]
else:
    collection = client["campomaq_test"]["cm_ventas_prods"]

# Read all documents from the collection
cursor = collection.find({})
df = pd.DataFrame(list(cursor))


In [14]:
df_sales_prods = df.copy()
df_sales_prods = df_sales_prods[df_sales_prods['invoice_date'].dt.year > 2023]
df_sales_prods = df_sales_prods[~df_sales_prods["category_name"].str.lower().str.contains("servicios", na=False)]
df_sales_prods = df_sales_prods[~df_sales_prods["product_name"].str.lower().str.contains("varios", na=False)]


In [31]:
rules_df = pd.read_csv("product_rules.csv")

In [ ]:
import pandas as pd
import numpy as np

# -------------------
# Step 1: Aggregate by product
# -------------------
df_grouped = df_sales_prods.groupby(
    ["product_code", "product_name", "brand_name", "category_name"], as_index=False
).agg({
    "quantity": "sum",
    "sale_without_iva": "sum",
    "total_cost": "sum"
})

# Profit
df_grouped["profit"] = df_grouped["sale_without_iva"] - df_grouped["total_cost"]

# Replace negative or zero profit with a small value
df_grouped["profit"] = df_grouped["profit"].apply(lambda x: x if x > 0 else 0.01)

# -------------------
# Step 2: Low-value and boost product flag
# -------------------
low_value_words = rules_df.query("rule_type == 'word' and category == 'low_value'")["rule_value"].str.lower().tolist()
low_value_codes = rules_df.query("rule_type == 'code' and category == 'low_value'")["rule_value"].tolist()

boost_words = rules_df.query("rule_type == 'word' and category == 'boost'")["rule_value"].str.lower().tolist()
boost_codes = rules_df.query("rule_type == 'code' and category == 'boost'")["rule_value"].tolist()

# -------------------
# Apply low-value flag
# -------------------
def is_low_value(row):
    name = row["product_name"].lower()
    code = row["product_code"]
    if any(word in name for word in low_value_words):
        return 1
    if code in low_value_codes:
        return 1
    return 0

# -------------------
# Apply strategic boost
# -------------------
def has_boost(row):
    name = row["product_name"].lower()
    code = row["product_code"]
    if any(word in name for word in boost_words):
        return 1
    if code in boost_codes:
        return 1
    return 0

df_grouped["main_boost"] = df_grouped.apply(has_boost, axis=1)
df_grouped["low_value_flag"] = df_grouped.apply(is_low_value, axis=1)


# -------------------
# Step 3: Normalization
# -------------------
# Quantity normalization
df_grouped["quantity_norm"] = (
    (df_grouped["quantity"] - df_grouped["quantity"].min()) /
    (df_grouped["quantity"].max() - df_grouped["quantity"].min())
)

# Penalize low-value items
df_grouped.loc[df_grouped["low_value_flag"] == 1, "quantity_norm"] *= 0.5

# Profit normalization
df_grouped["profit_norm"] = (
    (df_grouped["profit"] - df_grouped["profit"].min()) /
    (df_grouped["profit"].max() - df_grouped["profit"].min())
)


# -------------------
# Step 4: Popularity Index
# -------------------
df_grouped["popularity"] = (
    0.64 * df_grouped["quantity_norm"] +   # sales volume
    0.355 * df_grouped["profit_norm"] +     # profitability
    0.005 * df_grouped["main_boost"]        # strategic push
)

# -------------------
# Step 6: Sort products by popularity
# -------------------
df_grouped = df_grouped.sort_values("popularity", ascending=False)

# Check top results
# print(df_grouped[["product_name", "quantity", "profit", "popularity"]].head(15))


In [18]:
pd.set_option('display.max_rows', 500)

In [34]:
df_grouped.sort_values("popularity", ascending=False, inplace=True)
df_grouped.head(100)

,product_code,product_name,brand_name,category_name,quantity,sale_without_iva,total_cost,profit,main_boost,low_value_flag,quantity_norm,profit_norm,popularity
1131,408-0009SI,MOTOCULTOR TF545DE ARRANQUE ELECTRICO,HUSQVARNA,MOTOCULTORES,29,90930.000000,4.901309e+04,41916.906667,1,0,0.000306,1.000000,0.355184
2006,700M-017,NYLON 3.3MM OLEO PARA GUADAÑAS MARUYAMA,VARIOS MOTORES,REPUESTOS MOTOGUADANAS,91504,15452.319011,8.060341e+06,0.010000,0,1,0.500000,0.000000,0.300000
2121,700M-157,NYLON NEGRO 3.3MM OLEO,VARIOS MOTORES,VARIOS,69893,14489.830400,1.050231e+04,3987.523178,0,1,0.381911,0.095129,0.262442
1580,700-0001,MOTOCULTOR TKC-450,MITSUBISHI,MOTOCULTORES,9,48300.000000,2.420779e+04,24092.209709,0,0,0.000087,0.574761,0.201219
1417,440-0015,CADENA X ROLLO H42 (73VL).100.3/8 503-30-60-01...,HUSQVARNA,ACCESORIOS MOTOSIERRAS,14038,6956.167707,4.885180e+03,2070.987824,0,0,0.153405,0.049407,0.109335
1428,440-0048,"CADENA X ROLLO H52 72RD),3/8"", 1 GUIA 581-62-6...",HUSQVARNA,ACCESORIOS MOTOSIERRAS,14587,6552.344401,3.778069e+06,0.010000,0,0,0.159405,0.000000,0.095643
1813,700B-205,MANGUERA DE FUMIGACION NEGRA CON ROJO ITALIANA...,VARIOS BOMBAS,ACCESORIOS FUMIGACION,163,35079.911715,2.575400e+04,9325.911715,0,0,0.001770,0.222486,0.078932
1686,700B-003SI,DISCO C35,VARIOS BOMBAS,ACCESORIOS FUMIGACION,9899,31029.572500,2.565817e+04,5371.403472,0,1,0.054086,0.128144,0.077302
1776,700B-125SI,MANGUERA DE FUMIGACION 8.5MM CASAMOTO,VARIOS BOMBAS,ACCESORIOS FUMIGACION,221,19808.000000,1.082900e+04,8979.000000,0,0,0.002404,0.214209,0.076416
2614,S000066,TRABAJO DE TORNO,TORNO,TORNO,334,7924.304348,0.000000e+00,7924.304348,0,0,0.003639,0.189048,0.068350


In [35]:
if cm_env == 'prod':
    collection = client["campomaq"]["cm_top_sales_products"]
else:
    collection = client["campomaq_test"]["cm_top_sales_products"]
    
for record in df_grouped.to_dict(orient="records"):
    # Use product_id as unique key, update if exists, insert if not
    collection.update_one(
        {"product_code": record["product_code"]},
        {"$set": record},
        upsert=True
    )

print(f"✅ Upserted {len(df_grouped)} products into MongoDB.")

✅ Upserted 2617 products into MongoDB.


In [ ]:
# if cm_env == 'prod':
#     collection = client["campomaq"]["cm_top_sales_products"]
# else:
#     collection = client["campomaq_test"]["cm_top_sales_products"]

# if not df_grouped.empty:
#     # Convert DataFrame to JSON
#     records = df_grouped.to_dict(orient="records")
#     collection.insert_many(records)
#     print(f"✅ Inserted {len(records)} records into MongoDB")

# else:
#     print("No new data to insert.")

✅ Inserted 2611 records into MongoDB
